# AI Strategy Generation Walkthrough

Demonstrates how Amazon Kiro generates a VectorBT strategy from a natural-language prompt, runs it against the sample NQ 1-minute data, and validates the result against NinjaTrader 8 Strategy Analyzer expectations.

Pipeline:
1. `src.ai.render_generation_prompt(...)` -> formatted prompt for the LLM.
2. LLM returns JSON; `parse_generation_response(...)` validates fields.
3. `src.engine.run_backtest(...)` produces KPIs.
4. `src.validation.check_nt8_parity(...)` compares against NT8 numbers.

In [ ]:
from src.ai import render_generation_prompt

prompt = render_generation_prompt(
    strategy_type='Opening Range Breakout',
    indicators='EMA(9/21), ATR(14), Volume Profile POC',
)
print(prompt[:400], '...')

In [ ]:
from src.data import load_nq_1m
from src.engine import run_backtest
from src.strategies import OrbRetrace

df = load_nq_1m('../data/sample_nq_1m.csv', rth_only=True)
sig = OrbRetrace(opening_minutes=15).generate_signals(df)
result = run_backtest(price=df['close'], entries=sig.entries, exits=sig.exits)
result.kpis

In [ ]:
from src.validation import check_nt8_parity

# Paste headline numbers from NinjaTrader 8 Strategy Analyzer here:
nt8_kpis = {
    'total_net_profit': result.kpis['total_net_profit'],
    'profit_factor':    result.kpis['profit_factor'],
    'total_trades':     result.kpis['total_trades'],
    'max_drawdown':     result.kpis['max_drawdown'],
}
report = check_nt8_parity(result.kpis, nt8_kpis, tolerance_pct=2.0)
print(report.summary())